# 01 — Prior Truncation (tSNPE)

En este tutorial exploramos el truncamiento secuencial del prior (tSNPE) y cómo usar `update_proposal` para generar proposals iterativamente.

In [ ]:
import shutil
import json
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.distributions import Uniform
import zbi
from zbi.utils import compute_bounding_box, TruncatedBoxPrior

%matplotlib inline

## 1. Setup del experimento

Reusamos el mismo simulador y embedding del tutorial 00.

In [ ]:
class SimuladorLineal(zbi.Simulator):
    def __init__(self):
        super().__init__()
        self.x_grid = torch.linspace(-5, 5, 100)

    def simulate(self, theta: torch.Tensor, seed: int | None = None) -> torch.Tensor:
        if seed is not None:
            torch.manual_seed(seed)
        a, b = theta[0].item(), theta[1].item()
        y = a * self.x_grid + b + 0.1 * torch.randn(100)
        return y


class EmbeddingNet(nn.Module):
    def __init__(self, dim_in: int = 100, dim_out: int = 5):
        super().__init__()
        self.net = nn.Linear(dim_in, dim_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
run_dir = "runs/tutorial_01"

if os.path.exists(run_dir):
    shutil.rmtree(run_dir)

x_o = zbi.simulate_obs(SimuladorLineal(), theta_true=(1.0, 0.5), seed=42)

zbi.init(
    run_dir=run_dir,
    x_o=x_o,
    simulator_class=SimuladorLineal,
    embedding_class=EmbeddingNet,
    embedding_kwargs=dict(dim_in=100, dim_out=5),
    prior_low=(-3.0, -3.0),
    prior_high=(3.0, 3.0),
    dim_theta=2,
    dim_x=100,
)
print("Experimento inicializado")

## 2. Ronda 0: Prior completo

Entrenamos con el prior Uniforme completo en $[-3, 3]^2$.

In [ ]:
zbi.simulate(run_dir=run_dir, round=0, n_sims=500, seed=1)
zbi.train(run_dir=run_dir, round=0, n_sims=500, max_epochs=200, stop_after_epochs=20)
print("Ronda 0 completada")

## 3. Generar proposal con `update_proposal`

`update_proposal` carga el checkpoint, samplea la posterior, y computa bounds truncados usando `compute_bounding_box`.

El algoritmo de truncamiento:
1. Samplea $N$ muestras de la posterior $\{\theta^{(i)}\}$
2. Calcula $\log p(\theta^{(i)} \mid x_o)$ para cada muestra
3. Selecciona muestras con $\log p > \max(\log p) + \log(\text{threshold})$
4. La nueva caja es `[min, max]` por dimensión de las muestras seleccionadas

El `threshold` controla qué tan conservador es el truncamiento:
- `threshold=1e-6`: muy conservador (caja más grande)
- `threshold=1e-3`: moderado
- `threshold=0.1`: agresivo (caja más pequeña, riesgo de excluir el verdadero valor)

In [ ]:
zbi.update_proposal(
    run_dir=run_dir,
    checkpoint="round_00000_500.pt",
    threshold=1e-6,
)

with open(f"{run_dir}/proposal.json") as f:
    proposal = json.load(f)
print(f"Bounds low:  {proposal['bounds']['low']}")
print(f"Bounds high: {proposal['bounds']['high']}")
print(f"Guardado en proposals/round_0.json")

## 4. Visualizar el truncamiento

Comparación de bounding boxes para diferentes thresholds.

In [ ]:
# Samplear de la posterior
samples = zbi.sample_model(
    run_dir=run_dir,
    checkpoint="round_00000_500.pt",
    n_samples=10_000,
)

# Compute bounding boxes para diferentes thresholds
prior_original = Uniform(torch.tensor([-3.0, -3.0]), torch.tensor([3.0, 3.0]))
original_bounds = torch.stack([prior_original.low, prior_original.high])

# Log-probs de las muestras
from zbi.pipeline.checkpoints import _load_maf_from_run
maf = _load_maf_from_run(run_dir, "round_00000_500.pt")
maf.eval()
with torch.no_grad():
    theta_t = torch.from_numpy(samples).float()
    x_o_batch = x_o.expand(theta_t.shape[0], -1)
    log_probs = maf.forward(theta_t, x_o_batch)

thresholds = [1e-6, 1e-3, 0.01, 0.1]
boxes = {}
for th in thresholds:
    bounds = compute_bounding_box(theta_t, log_probs, th, original_bounds)
    boxes[th] = bounds

# Plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.3, color="C1")
ax.scatter(1.0, 0.5, marker="*", s=200, color="red", zorder=5, label=r"$\theta_{\mathrm{true}}$")

colors = ["C0", "C1", "C2", "C3"]
for i, (th, bounds) in enumerate(boxes.items()):
    low, high = bounds[0], bounds[1]
    w, h = high[0] - low[0], high[1] - low[1]
    rect = plt.Rectangle(low.numpy(), w, h, fill=False,
                         edgecolor=colors[i], lw=2, ls="--",
                         label=f"threshold={th:.0e}")
    ax.add_patch(rect)

ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.legend(fontsize=10)
ax.set_title("Bounding boxes para diferentes thresholds")
plt.tight_layout()
plt.show()

## 5. Ronda 1: Prior truncado

Cuando llamas `simulate` para `round > 0`, automáticamente lee `proposal.json` y construye un `TruncatedBoxPrior` con los bounds guardados.

In [ ]:
zbi.simulate(run_dir=run_dir, round=1, n_sims=500, seed=2)
zbi.train(run_dir=run_dir, round=1, n_sims=500, max_epochs=200, stop_after_epochs=20)
zbi.update_proposal(run_dir=run_dir, checkpoint="round_00001_500.pt", threshold=1e-6)
print("Ronda 1 completada")

In [ ]:
with open(f"{run_dir}/proposal.json") as f:
    proposal = json.load(f)
print(f"Bounds ronda 1:")
print(f"  low:  {proposal['bounds']['low']}")
print(f"  high: {proposal['bounds']['high']}")

print(f"\nProposals guardados:")
print(f"  {sorted(os.listdir(f'{run_dir}/proposals'))}")

## 6. Comparar rondas

In [ ]:
samples_r0 = zbi.sample_model(run_dir, "round_00000_500.pt", n_samples=10_000)
samples_r1 = zbi.sample_model(run_dir, "round_00001_500.pt", n_samples=10_000)

g = zbi.plot_ppc(
    all_samples=[samples_r0, samples_r1],
    param_names=[r'\theta_0', r'\theta_1'],
    true_parameter=[1.0, 0.5],
    sample_labels=["Round 0 (prior completo)", "Round 1 (prior truncado)"],
    sample_colors=["C0", "C1"],
    filled=[True, False],
    output_path=f"{run_dir}/plots/comparison_r0_r1.pdf",
)
print(f"Plot guardado en {run_dir}/plots/comparison_r0_r1.pdf")

## Resumen

| Concepto | Descripción |
|----------|-------------|
| `threshold` | Controla la conservación del truncamiento (menor = más conservador) |
| `proposal.json` | Guarda bounds, prior_meta, y checkpoint usado |
| `proposals/round_N.json` | Historial de proposals por ronda |
| `TruncatedBoxPrior` | Prior construido automáticamente por `simulate` para round > 0 |

En el siguiente tutorial veremos inferencia marginal con `interest_dims`.